# Assignment 1 Part B - Phoneme Recognition

Welcome to the second part of the assignment! Make sure to read the writeup before beginning work here.

## Setup/Importing

In [1]:
# @title
# TODO: Run this cell and follow instructions to connect this notebook to Google Drive
# Additional guidance: https://colab.research.google.com/notebooks/io.ipynb

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not on google drive")

Mounted at /content/drive


In [2]:
# @title
# TODO: Change directories ("cd") to the folder containing your
# notebook and data folder by replacing the filepath below
%cd /content/drive/MyDrive/Colab_Notebooks/assignment1b/

/content/drive/MyDrive/Colab_Notebooks/assignment1b


### Download and Unzip Data

To resolve the `ImportError`, you need to download `data1pb.zip` which contains `utils.py`. Please replace `'YOUR_DOWNLOAD_URL_HERE'` with the actual download link for `data1pb.zip`.

In [ ]:
# TODO: Replace 'YOUR_DOWNLOAD_URL_HERE' with the correct URL for pa1b_starter.zip
# Then, run this cell to download the data.
!wget 'https://mo-pcco.s3.us-east-1.amazonaws.com/cmu-dele/assignments/pa1b_starter.zip' -O pa1b_starter.zip

--2026-05-31 00:14:26--  https://mo-pcco.s3.us-east-1.amazonaws.com/cmu-dele/assignments/pa1b_starter.zip
Resolving mo-pcco.s3.us-east-1.amazonaws.com (mo-pcco.s3.us-east-1.amazonaws.com)... 16.15.183.222, 52.217.121.170, 16.15.191.153, ...
Connecting to mo-pcco.s3.us-east-1.amazonaws.com (mo-pcco.s3.us-east-1.amazonaws.com)|16.15.183.222|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1469338 (1.4M) [application/zip]
Saving to: ‘pa1b_starter.zip’

pa1b_starter.zip    100%[===================>]   1.40M  3.18MB/s    in 0.4s    

2026-05-31 00:14:27 (3.18 MB/s) - ‘pa1b_starter.zip’ saved [1469338/1469338]



In [ ]:
# TODO: Replace 'YOUR_DOWNLOAD_URL_HERE' with the correct URL for data1pb.zip
# Then, run this cell to download the data.
!wget 'https://cmu-dele-leaderboard-us-east-2-003014019879.s3.us-east-2.amazonaws.com/colab/pa1b/data1pb.zip' -O data1pb.zip

--2026-05-30 05:34:47--  https://cmu-dele-leaderboard-us-east-2-003014019879.s3.us-east-2.amazonaws.com/colab/pa1b/data1pb.zip
Resolving cmu-dele-leaderboard-us-east-2-003014019879.s3.us-east-2.amazonaws.com (cmu-dele-leaderboard-us-east-2-003014019879.s3.us-east-2.amazonaws.com)... 3.5.89.56, 3.5.89.104, 52.219.108.82, ...
Connecting to cmu-dele-leaderboard-us-east-2-003014019879.s3.us-east-2.amazonaws.com (cmu-dele-leaderboard-us-east-2-003014019879.s3.us-east-2.amazonaws.com)|3.5.89.56|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8917643877 (8.3G) [application/zip]
Saving to: ‘data1pb.zip’

data1pb.zip         100%[===================>]   8.30G  49.1MB/s    in 3m 41s  

2026-05-30 05:38:28 (38.5 MB/s) - ‘data1pb.zip’ saved [8917643877/8917643877]



In [ ]:
# TODO: Run this cell to unzip the utils.py.
# -o: Overwrites existing files on the disk without prompting for confirmation.
!unzip -o pa1b_starter.zip utils.py

Archive:  pa1b_starter.zip
  inflating: utils.py                


In [ ]:
# TODO: Run this cell to unzip the downloaded data.
# -o: Overwrites existing files on the disk without prompting for confirmation.
!unzip -o data1pb.zip

Archive:  data1pb.zip
  inflating: data/phonemes.txt       
  inflating: data/sample.csv         
  inflating: data/train_labels.npy   
  inflating: data/val_labels.npy     
  inflating: data/val.npy            
  inflating: data/test.npy           
  inflating: data/pa1b_solution.csv  
  inflating: data/train.npy          
  inflating: data/pa1b_sample.csv    
 extracting: data/11785-spring2021-hw1p2.zip  


After successfully downloading and unzipping the file, please re-run cell `3f834204` to confirm `local_utils.py` is accessible, and then re-run cell `JAqExxnVDdtm`.

In [3]:
import sys
import os

# Add the current working directory to sys.path to prioritize local modules
sys.path.insert(0, os.getcwd())

print("Current working directory added to sys.path.")
# Verify that utils.py is now accessible
try:
    from local_utils import num_ms
    print("Successfully imported num_ms from local_utils.py")
except ImportError:
    print("Still unable to import num_ms from local_utils.py.")

Current working directory added to sys.path.
Successfully imported num_ms from local_utils.py


Let's check if `local_utils.py` is actually in the current directory and what its contents are.

In [4]:
# List files in the current directory to verify utils.py exists
!ls -F

data/	     local_utils.py	    pa1b_starter.zip  writup/
data1pb.zip  pa1b_assignment.ipynb  __pycache__/


In [ ]:
# Display the content of local_utils.py to check for num_ms definition
!cat local_utils.py

import bisect
import csv
from datetime import datetime
import os

import matplotlib.pyplot as plt
import numpy as np
import torch


class KContextSpectrograms(torch.utils.data.Dataset):
    """Preprocesses dataset and returns frame surrounded by K context frames on both sides.
    
    Inherits from PyTorch Dataset class.
    
    Example:
    >>> dev_dataset = KContextSpectrograms(data/dev.npy, k=0)

    Args:
        data_path (str): Path to data file (i.e. "data/dev.npy")
        labels_path (str): Path to labels file (i.e. "data/dev_labels.npy")
        k (int): How many context frames to add to each side of the frame 
    """
    def __init__(self, data_path, labels_path=None, k=0):
        # Load in data (and labels, if available)
        data = np.load(data_path, allow_pickle=True)
        labels = np.load(labels_path, allow_pickle=True) if labels_path else None
            
        # Get the total number of frames in the dataset
        self.total_frames = np.sum([len(utterance

In [5]:
# Run this cell to import packages
import numpy as np
import torch
import torch.nn as nn
from tqdm.notebook import tqdm

import os

# %load_ext autoreload
# %autoreload 2

### Auto-detect if GPU is available

In [6]:
# TODO: Run this cell to automatically detect if GPU is available.
# Output should be 'cuda' if you are expecting to be on a GPU
# If the output is equal to 'cpu', click on 'Runtime' in the top menu, then 'Change Runtime Type', and select 'GPU'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

cuda


# Section 1: `Dataset`/`DataLoader`

When working with any dataset in `torch`, you'll almost always work with a `Dataset` and `DataLoader` object. Here's an overview of what they usually do:

`torch.utils.data.Dataset`
- Stores dataset (usually a single tensor or list of tensors) inside the object
    - Happens in `__init__()` function
- Defines how many observations are in the dataset
    - In `__len__()`
- Defines how to retrieve a single observation from the dataset given its index from 0 inclusive to `__len__()` exclusive
    - In `__getitem__()`


`torch.utils.data.DataLoader`
- Queries and batches observations from an initialized `Dataset`
- If `shuffle=True`, shuffles dataset for you every epoch (do this for training, not validation / testing)
- Handles basic multiprocessing

Some specialized datasets (like in this assignment) usually need a custom `Dataset` class. However, for popular datasets, there are often existing implementations, like those found [here](https://pytorch.org/vision/stable/datasets.html).

## Question 1.1: Initialize Dataset
Fortunately we've already written the custom `Dataset` for you.

Open `utils.py`, find `KContextSpectrograms`, and read through it to understand how it works and what initialization parameters are required. Afterwards, complete and run each cell marked with `# TODO`.

In [7]:
from local_utils import num_ms

# Specify the desired number of context frames to concatenate to each side of your target,
# then run this cell to preview how many milliseconds will be covered with your selected `k`
k = 30

print(f"# milliseconds covered with context {k}:", num_ms(k))

# milliseconds covered with context 30: 635


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# Initialize dataset objects for training, validation, and testing.
from local_utils import KContextSpectrograms

# Assuming 'k' is defined in a previous cell, e.g., JAqExxnVDdtm
# k = 30

# KContextSpectrograms: This line imports the custom Dataset class,
# which is designed to handle spectrogram data with 'k' context frames.
train_dataset = KContextSpectrograms(data_path="data/train.npy", labels_path="data/train_labels.npy", k=k)
val_dataset = KContextSpectrograms(data_path="data/val.npy", labels_path="data/val_labels.npy", k=k)
test_dataset = KContextSpectrograms(data_path="data/test.npy", labels_path=None, k=k)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 18482968
Validation dataset size: 1559057
Test dataset size: 1618835


## Question 1.2: Initialize DataLoaders
Although `Dataset`s frequently need custom implementations, `DataLoader`s are usually standard.

Use [this documentation](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) to guide you on implementing the following instructions:

- Specify an adequate batch size (see writeup)
- Initialize training dataloader with batch size, pinned memory, number of workers, and with shuffling.
- Initialize validation dataloader and test dataloader with your batch size, pinned memory, number of workers, and WITHOUT shuffling.
    - We don't shuffle val because we're just calculating the accuracy on every observation, so shuffling just slows things down without mattering
    - We don't shuffle test because we need to export our predictions in the correct order
    - We use num_workers equal to cpu count to prepare data in parallel
    - We pin memory to speed up data transfer

**Hint**: For examples of initializing `Dataset`s/`DataLoader`s, see [here](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html).

In [10]:
import os
import torch

# Specify how many observations should go in each batch
batch_size = 64  # A common default batch size, you can adjust this

# pass num_workers into each dataloader
num_workers = os.cpu_count()

# Initialize `Dataloader` objects for training, validation, and testing.
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=num_workers)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=num_workers)
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True, num_workers=num_workers)

print(f"Train dataloader batches: {len(train_dataloader)}")
print(f"Validation dataloader batches: {len(val_dataloader)}")
print(f"Test dataloader batches: {len(test_dataloader)}")

Train dataloader batches: 288797
Validation dataloader batches: 24361
Test dataloader batches: 25295


# Section 2: Training/Validation/Prediction Routines

Below are a few given methods that should look familiar, but there are some important differences you should try to spot. Read these through carefully.

In [11]:
def train(model, optimizer, scheduler, train_dataloader, val_dataloader, num_epochs, patience=5, min_delta=0.001):
    """[Given] Trains and validates network for `num_epochs`

    Args:
        model (nn.Sequential): Initialized network, stored in an `nn.Sequential` object.
        optimizer (optim.Optimizer): Initialized optimizer like `optim.SGD` or `optim.Adam`
        scheduler (optim.lr_scheduler): Initialized scheduler like `optim.lr_scheduler.ReduceLROnPlateau` (or None)
        train_dataloader (torch.utils.data.DataLoader): Initialized training dataloader
        val_dataloader (torch.utils.data.DataLoader): Initialized validation dataloader
        num_epochs (int): # epochs to train for
        patience (int): Number of epochs to wait for improvement before stopping.
        min_delta (float): Minimum change in validation accuracy to qualify as an improvement.
    Returns:
        list, list: losses is the loss per every batch, val_accuracies is the val accuracy per epoch
    """
    losses = []
    val_accuracies = []
    best_val_accuracy = -float('inf')
    epochs_no_improve = 0

    for e in range(num_epochs):
        # No need to manually reshuffle; `Dataloader` handles that for you!
        # Train model for one epoch
        epoch_losses = train_epoch(model, optimizer, train_dataloader, scheduler)
        losses.extend(epoch_losses)

        # Evaluate model on validation set, track accuracy
        val_accuracy = validate(model, val_dataloader)
        print("Validation Accuracy", 100 * val_accuracy)
        val_accuracies.append(val_accuracy)

        # Early stopping logic
        if val_accuracy > best_val_accuracy + min_delta:
            best_val_accuracy = val_accuracy
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {e + 1} epochs due to no improvement in validation accuracy for {patience} epochs.")
            break

    return losses, val_accuracies

In [12]:
# [Given] ALREADY COMPLETED, SHOWN JUST FOR YOUR REFERENCE
def validate(model, dataloader):
    """[Given] Evaluates network and calculates accuracy for a full validation dataset.

    Args:
        model (nn.Sequential): Your initialized network, stored in a `Sequential` object.
        dataloader (torch.utils.data.DataLoader): Initialized validation dataloader

    Returns:
        float: Accuracy rate for entire val set.
    """
    # Set model to evaluate mode (train mode is `.train()`)
    model.eval()

    total_correct = 0
    # Run loop with `tqdm` progress bar
    for i, (data, labels) in tqdm(enumerate(dataloader), total=len(dataloader), desc="Validate", colour="orange"):
        # Put tensors on specified device (GPU or CPU)
        data, labels = data.to(DEVICE), labels.to(DEVICE)
        logits = model(data)
        num_correct = (logits.argmax(axis=1) == labels).cpu().numpy().sum()
        total_correct += num_correct
    return total_correct / len(dataloader.dataset)

## Question 2.1: `train_epoch()`

Now to write the training routine of a single epoch.

See the `validate()` method especially for hints, section 4 [here](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) is also a good reference.

```
def train_epoch():
    set_model_to_train_mode()
    loss_function = create_loss_function_object() # cross entropy!
    for (data, labels) in tqdm(dataloader):
        data, labels = put_tensors_on_appropriate_device(DEVICE, data, labels) # See val method for hint
        reset_gradients_to_zero()
        logits = forward_pass_through_model(model, data)
        loss = loss_function(logits, labels)
        run_backprop() # look up how torch does this; it's slightly different from what you did in part A
        update_model_params() # using the optimizer
        store_loss_value(loss)
    return loss_values
```

In [13]:
def train_epoch(model, optimizer, dataloader, scheduler=None):
    """Train model for one epoch.

    Args:
        model (nn.Sequential): Initialized network, stored in a `nn.Sequential` object.
        optimizer (optim.Optimizer): Initialized optimizer like `optim.SGD` or `optim.Adam`
        dataloader (torch.utils.data.DataLoader): Initialized training dataloader
        scheduler (optim.lr_scheduler): Optional scheduler if you want it


    Returns:
        list: Loss value of each batch for this epoch.
    """
    # Append loss values to this list.
    loss_per_batch = []

    # Set model to train mode
    model.train()
    """
    In this mode:
    1) dropout is active, and
    2) batch normalization updates its running statistics.
    """

    # Define the loss function (Cross Entropy for classification)
    loss_fn = torch.nn.CrossEntropyLoss()

    # Run loop with `tqdm` progress bar
    for i, (data, labels) in tqdm(enumerate(dataloader), total=len(dataloader), desc="Train epoch", colour="blue"):
        # Put tensors on specified device (GPU or CPU)
        data, labels = data.to(DEVICE), labels.to(DEVICE)

        # Reset gradients to zero (PyTorch accumulates gradients by default)
        optimizer.zero_grad()

        # Forward pass through the model
        logits = model(data)    # computes the raw, unnormalized scores (logits) for each phoneme class.

        # Calculate the loss
        loss = loss_fn(logits, labels)

        # Run backpropagation (PyTorch automatically calculates the gradients of the loss with respect to all parameters that have requires_grad=True)
        loss.backward()

        # Update model parameters (Updates the model's parameters using the computed gradients and the chosen optimization algorithm (e.g., SGD, Adam))
        optimizer.step()  # The optimizer adjusts the weights in the direction that minimizes the loss

        # Store the loss value for the batch
        loss_per_batch.append(loss.item())

    # (Feel free to change) If scheduler, determine if we should change LR based on some metric
    if scheduler is not None:
        scheduler.step(sum(loss_per_batch)) # This assumes ReduceLROnPlateau; the choice of using sum is fairly arbitrary.

    return loss_per_batch

## Question 2.2: `predict()`

This method is very similar to the `validate()` method we gave above. It's used to generate predictions for all observations in the test dataset.

You can assume that the dataloader is NOT shuffled. Each batch you receive will have `batch_size` number of observations (no labels), and you want to `extend` a list containing your previous predictions. The end result will be a 1-dimensional list containing integers, the same length as the test dataset.

In [14]:
def predict(model, dataloader):
    """Generates predictions for the test dataset.
    This function iterates through the test dataset in batches,
    obtains the model's predictions, and
    compiles them into a single list.

    Args:
        model (nn.Sequential): Your initialized network, stored in a `Sequential` object.
        dataloader (torch.utils.data.DataLoader): Initialized test dataloader

    Returns:
        list: should be same length as test dataset, and containing ints (or numpy integers)
    """

    # Set model to evaluation mode
    model.eval()
    """
    In this mode,
    1) dropout layers are turned off, and
    2) batch normalization layers use their running mean and variance (instead of batch statistics).
    This ensures consistent predictions.
    """

    # An empty list is initialized to store all the predictions from the test dataset.
    preds = []
    for i, data in tqdm(enumerate(dataloader), total=len(dataloader), desc="Predict", colour="green"):
        # The input data (spectrograms) from the current batch is moved to the specified device (GPU or CPU)
        data = data.to(DEVICE)

        # Forward pass through the model and get the logits
        logits = model(data)  # utputs raw, unnormalized scores (logits) for each possible phoneme class.

        # Get your batched predictions from your logits (class with highest score)
        # argmax(dim=1) finds the index of the class with the highest logit score.
        # .cpu(): The predicted class indices, which are currently on the DEVICE (e.g., GPU), are moved back to the CPU.
        # .tolist(): The PyTorch tensor of predicted indices is converted into a standard Python list.
        predicted = logits.argmax(dim=1).cpu().tolist()

        # Extend your preds list with them
        preds.extend(predicted)   # extend is used here instead of append to add all elements of the predicted list individually, resulting in a single flat list of predictions.
    return preds

# Section 3: Training

You're done with the major coding work! Now just to initialize your model/optimizer and begin training.

### Determining Model Parameters

Before initializing the model, we need to know the input feature dimension and the number of output classes (phonemes).

The `KContextSpectrograms` dataset prepares input frames by flattening `(2 * k + 1)` context frames, each with a certain number of features.

Assuming a common mel-spectrogram feature dimension of 40, and with `k=30`, the input dimension will be `(2 * 30 + 1) * 40 = 2440`.

We also need to determine the number of phonemes from `data/phonemes.txt` for the output layer.

In [15]:
# Determine the number of phoneme classes from data/phonemes.txt
with open('data/phonemes.txt', 'r') as f:
    num_phonemes = len(f.readlines())

print(f"Number of phoneme classes: {num_phonemes}")

# Based on KContextSpectrograms and k=30, assuming 40 features per frame:
input_feature_dim = (2 * k + 1) * 40 # 61 frames * 40 features/frame
print(f"Input feature dimension: {input_feature_dim}")

Number of phoneme classes: 71
Input feature dimension: 2440


Now that we have the `num_phonemes` and `input_feature_dim`, we can define the model, optimizer, and scheduler.

In [16]:
# Initialize your model here; an EXAMPLE model is provided below
# Assuming input_feature_dim and num_phonemes are determined from previous steps
# A simple MLP model with Batch Normalization for improved training stability and performance:
model = nn.Sequential(
    nn.Linear(input_feature_dim, 1024), # First hidden layer (increased neurons)
    nn.BatchNorm1d(1024), # Batch Normalization
    nn.ReLU(),
    nn.Dropout(0.5), # Add dropout for regularization

    nn.Linear(1024, 512), # Second hidden layer
    nn.BatchNorm1d(512), # Batch Normalization
    nn.ReLU(),
    nn.Dropout(0.5),

    nn.Linear(512, 256), # Third hidden layer
    nn.BatchNorm1d(256), # Batch Normalization
    nn.ReLU(),
    nn.Dropout(0.5),

    nn.Linear(256, num_phonemes) # Output layer: one logit per phoneme class
)

"""
To improve the model architecture,
1) added Batch Normalization layers and
2) added an extra hidden layer
3) increased the model's capacity with increased neuron counts (1024, 512, 256) for more capacity for the model to learn complex patterns.
Batch Normalization can stabilize and accelerate training by normalizing the inputs to
each layer, which can help with deeper networks.
"""

# put model on `DEVICE`
model = model.to(DEVICE)

# Initialize the optimizer
# Adam is a popular choice for its adaptive learning rate properties
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

#  (optional) Initialize scheduler
# ReduceLROnPlateau is a good general-purpose scheduler that reduces LR when a metric stops improving
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

### Convolutional Neural Network (CNN) Architecture

For spectrogram data, CNNs are generally more effective than simple Multi-Layer Perceptrons (MLPs). They excel at automatically learning hierarchical features and are robust to minor shifts in the input.

Here's the proposed CNN architecture:

1.  **Input Reshaping**: The flattened input feature vector (`input_feature_dim`) from `KContextSpectrograms` (which is `(2*k+1) * features_per_frame`) is reshaped into a 2D format `(batch_size, 1, 2*k+1, features_per_frame)`. The `1` represents a single channel (like a grayscale image).
2.  **Convolutional Blocks**: Multiple `nn.Conv2d` layers are used. Each `Conv2d` layer is followed by `nn.BatchNorm2d`, `nn.ReLU` activation, `nn.MaxPool2d` for downsampling, and `nn.Dropout` for regularization.
    *   The `kernel_size=(3,3)` allows the convolutions to capture local patterns across both time and frequency axes.
    *   `MaxPool2d(kernel_size=(2,2))` progressively reduces the spatial dimensions, making the model more robust to small variations and reducing the number of parameters.
3.  **Flattening**: After the convolutional layers, the 2D feature maps are flattened into a 1D vector.
4.  **Fully Connected Layers**: This flattened vector is then passed through one or more `nn.Linear` (fully connected) layers, similar to the MLP, ending with an output layer that produces logits for each phoneme class.

This architecture leverages the spatial structure of spectrograms, allowing the model to learn more meaningful representations.

In [18]:
class PhonemeCNN(nn.Module):
    def __init__(self, input_feature_dim, num_phonemes, features_per_frame=40, k=30):
        super(PhonemeCNN, self).__init__()
        self.time_context_frames = 2 * k + 1 # e.g., 61 for k=30
        self.features_per_frame = features_per_frame # e.g., 40

        # Ensure the input_feature_dim matches our expected 2D shape
        assert input_feature_dim == self.time_context_frames * self.features_per_frame, \
            f"Input feature dim {input_feature_dim} does not match expected {self.time_context_frames * self.features_per_frame}"

        self.conv_layers = nn.Sequential(
            # Input: (batch_size, 1, time_context_frames, features_per_frame) -> (N, 1, 61, 40)
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=(3, 3), padding=(1, 1)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)), # Output: (N, 32, 30, 20) (ceil(61/2)=31, ceil(40/2)=20)
            nn.Dropout(0.25),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3, 3), padding=(1, 1)),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)), # Output: (N, 64, 15, 10) (ceil(31/2)=16, ceil(20/2)=10)
            nn.Dropout(0.25),
        )

        # Calculate the size of the flattened features after conv layers
        # Example: 64 channels * 16 height * 10 width = 10240
        # For k=30, time_context_frames=61, features_per_frame=40
        # After 2 MaxPool2d with kernel_size=(2,2), dimensions are roughly divided by 4
        # H_out = floor(H_in / 2), W_out = floor(W_in / 2)
        # First pool: (ceil(61/2)=31, ceil(40/2)=20)
        # Second pool: (ceil(31/2)=16, ceil(20/2)=10)

        # Manually calculate the output size after convolutions to ensure correctness
        # Create a dummy input to trace its size
        with torch.no_grad():
            dummy_input = torch.zeros(1, 1, self.time_context_frames, self.features_per_frame)
            conv_output_size = self.conv_layers(dummy_input).numel() // 1 # numel() gets total elements, divide by batch_size

        self.fc_layers = nn.Sequential(
            nn.Linear(conv_output_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_phonemes)
        )

    def forward(self, x):
        # Reshape input for Conv2d: (batch_size, 1, time_context_frames, features_per_frame)
        # x has shape (batch_size, input_feature_dim)
        x = x.view(-1, 1, self.time_context_frames, self.features_per_frame)
        x = self.conv_layers(x)
        x = torch.flatten(x, 1) # Flatten starting from dimension 1 (keep batch dim)
        x = self.fc_layers(x)
        return x

In [20]:
# Initialize your new CNN model
# Pass features_per_frame=40 (as assumed) and k (from earlier definition)
model = PhonemeCNN(input_feature_dim=input_feature_dim, num_phonemes=num_phonemes, features_per_frame=40, k=k)

# put model on `DEVICE`
model = model.to(DEVICE)

# Initialize the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Initialize scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

In [ ]:
# Call the training routine for some epochs (train)
num_epochs = 20
patience = 5 # Example patience value
min_delta = 0.001 # Example minimum delta for improvement

losses, val_accuracies = train(model, optimizer, scheduler, train_dataloader, val_dataloader, num_epochs, patience=patience, min_delta=min_delta)

Train epoch:   0%|          | 0/288797 [00:00<?, ?it/s]

Validate:   0%|          | 0/24361 [00:00<?, ?it/s]

Validation Accuracy 69.57898267991484


Train epoch:   0%|          | 0/288797 [00:00<?, ?it/s]

Validate:   0%|          | 0/24361 [00:00<?, ?it/s]

Validation Accuracy 70.50236136331128


Train epoch:   0%|          | 0/288797 [00:00<?, ?it/s]

Validate:   0%|          | 0/24361 [00:00<?, ?it/s]

Validation Accuracy 70.7412878425869


Train epoch:   0%|          | 0/288797 [00:00<?, ?it/s]

In [ ]:
# [GIVEN] You can plot your training loss progress using this function below.
from local_utils import plot_loss

plot_loss(losses, num_batches=len(train_dataloader), num_epochs=20)

# Section 4: Predict Test Dataset and Submission

If you're ready to submit, run these cells below and look for the output file generated in the `submissions/` folder (it will be time-stamped).

**NOTE:** The first row of the CSV should look like this:

`Id,Category`



*e.g.* `Id, deepLearner`

In [ ]:
preds = predict(model, test_dataloader)

In [ ]:
from local_utils import export_predictions_to_csv

export_predictions_to_csv(preds)

NameError: name 'preds' is not defined